# ComfyUI + Cloudflare Quick Tunnel smoke test

This notebook verifies the interactive browser path without downloading Qwen-Image-2.1.

It installs a pinned ComfyUI checkout and cloudflared inside Kaggle, starts ComfyUI on localhost, starts a temporary Cloudflare Quick Tunnel, and keeps the notebook alive for browser testing.

Security warning: Quick Tunnel exposes a public temporary URL without Cloudflare Access. Do not upload sensitive images or files during this test. Stop the notebook when finished.

In [ ]:
from __future__ import annotations

import json
import os
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen

WORKING = Path('/kaggle/working')
COMFY = WORKING / 'ComfyUI'
CLOUDFLARED = WORKING / 'cloudflared'
COMFY_LOG = WORKING / 'comfyui.log'
TUNNEL_LOG = WORKING / 'cloudflared.log'
STATUS = WORKING / 'tunnel_status.json'
RESULT_DIR = WORKING / 'tunnel_result'
COMFY_COMMIT = '93810483a4739a1588236919a3128d3070244146'
CLOUDFLARED_URL = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
PORT = 8188
MAX_CLOUDFLARED_BYTES = 150 * 1024 * 1024
RESULT_DIR.mkdir(parents=True, exist_ok=True)
COMFY_LOG.write_text('')
TUNNEL_LOG.write_text('')

status = {
    'stage': 'starting',
    'updated_at': datetime.now(timezone.utc).isoformat(),
    'comfy_commit': COMFY_COMMIT,
    'port': PORT,
    'tunnel_url': None,
    'comfy_ready': False,
    'tunnel_ready': False,
    'downloaded_bytes': 0,
}

def save_status():
    status['updated_at'] = datetime.now(timezone.utc).isoformat()
    STATUS.write_text(json.dumps(status, indent=2) + '\n')
    (RESULT_DIR / 'tunnel_status.json').write_text(json.dumps(status, indent=2) + '\n')
    print('STATUS', json.dumps(status, ensure_ascii=False), flush=True)

def run(command, cwd=None, timeout=None):
    print('RUN', ' '.join(str(part) for part in command), flush=True)
    return subprocess.run(
        [str(part) for part in command], cwd=str(cwd) if cwd else None,
        text=True, capture_output=True, timeout=timeout, check=False,
    )

def require_success(result, label):
    if result.returncode != 0:
        status['stage'] = f'failed: {label}'
        status[f'{label}_stdout'] = result.stdout[-8000:]
        status[f'{label}_stderr'] = result.stderr[-8000:]
        save_status()
        raise RuntimeError(f'{label} failed with {result.returncode}')

save_status()


In [ ]:
def download_cloudflared():
    status['stage'] = 'downloading_cloudflared'
    save_status()
    part = CLOUDFLARED.with_suffix('.part')
    part.unlink(missing_ok=True)
    request = Request(CLOUDFLARED_URL, headers={'User-Agent': 'kaggle-comfyui-tunnel-test/1.0'})
    total = 0
    with urlopen(request, timeout=45) as response, part.open('wb') as output:
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            total += len(chunk)
            if total > MAX_CLOUDFLARED_BYTES:
                raise RuntimeError('cloudflared download exceeded size limit')
            output.write(chunk)
            if total % (5 * 1024 * 1024) < 1024 * 1024:
                status['downloaded_bytes'] = total
                save_status()
    if total < 10 * 1024 * 1024:
        raise RuntimeError(f'cloudflared download is unexpectedly small: {total} bytes')
    with part.open('rb') as handle:
        if handle.read(4) != b'\x7fELF':
            raise RuntimeError('cloudflared is not an ELF executable')
    part.replace(CLOUDFLARED)
    CLOUDFLARED.chmod(0o755)
    status['downloaded_bytes'] = total
    status['cloudflared_size'] = CLOUDFLARED.stat().st_size
    save_status()

if not COMFY.exists():
    status['stage'] = 'cloning_comfyui'
    save_status()
    require_success(run(['git', 'init', str(COMFY)]), 'git init')
    require_success(run(['git', '-C', str(COMFY), 'remote', 'add', 'origin', 'https://github.com/Comfy-Org/ComfyUI.git']), 'git remote add')
    require_success(run(['git', '-C', str(COMFY), 'fetch', '--depth', '1', 'origin', COMFY_COMMIT], timeout=900), 'git fetch ComfyUI')
    require_success(run(['git', '-C', str(COMFY), 'checkout', '--detach', 'FETCH_HEAD']), 'git checkout ComfyUI')

if not (COMFY / 'main.py').exists():
    raise RuntimeError('ComfyUI main.py was not found after checkout')

status['stage'] = 'installing_comfyui_dependencies'
save_status()
require_success(run([
    sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-input',
    '-r', str(COMFY / 'requirements.txt'),
], timeout=1200), 'pip install requirements')

if not CLOUDFLARED.exists() or CLOUDFLARED.stat().st_size < 10 * 1024 * 1024:
    download_cloudflared()
else:
    status['cloudflared_size'] = CLOUDFLARED.stat().st_size
    save_status()

status['stage'] = 'starting_comfyui_and_tunnel'
save_status()


In [ ]:
def read_tail(path, limit=12000):
    return path.read_text(errors='replace')[-limit:] if path.exists() else ''

def stop_process(process):
    if process is None or process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=30)

comfy_process = None
tunnel_process = None
comfy_handle = COMFY_LOG.open('w', encoding='utf-8')
tunnel_handle = TUNNEL_LOG.open('w', encoding='utf-8')
try:
    comfy_process = subprocess.Popen(
        [sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', str(PORT), '--disable-auto-launch'],
        cwd=str(COMFY), stdout=comfy_handle, stderr=subprocess.STDOUT, text=True,
    )
    status['comfy_pid'] = comfy_process.pid
    save_status()
    deadline = time.time() + 180
    while time.time() < deadline:
        if comfy_process.poll() is not None:
            status['stage'] = 'failed_comfyui_exited'
            status['comfy_log_tail'] = read_tail(COMFY_LOG, 16000)
            save_status()
            raise RuntimeError('ComfyUI exited before /system_stats became ready')
        try:
            with urlopen(f'http://127.0.0.1:{PORT}/system_stats', timeout=3) as response:
                if response.status == 200:
                    status['comfy_ready'] = True
                    break
        except Exception:
            time.sleep(2)
    if not status['comfy_ready']:
        status['stage'] = 'failed_comfyui_timeout'
        status['comfy_log_tail'] = read_tail(COMFY_LOG, 16000)
        save_status()
        raise RuntimeError('ComfyUI did not become ready within 180 seconds')

    status['stage'] = 'starting_cloudflared'
    save_status()
    tunnel_process = subprocess.Popen(
        [str(CLOUDFLARED), 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'],
        stdout=tunnel_handle, stderr=subprocess.STDOUT, text=True,
    )
    status['tunnel_pid'] = tunnel_process.pid
    save_status()
    pattern = re.compile(r'https://[a-z0-9-]+\.trycloudflare\.com')
    deadline = time.time() + 180
    while time.time() < deadline:
        if tunnel_process.poll() is not None:
            status['stage'] = 'failed_cloudflared_exited'
            status['tunnel_log_tail'] = read_tail(TUNNEL_LOG, 16000)
            save_status()
            raise RuntimeError('cloudflared exited before producing a URL')
        match = pattern.search(read_tail(TUNNEL_LOG, 30000))
        if match:
            status['tunnel_ready'] = True
            status['tunnel_url'] = match.group(0)
            break
        time.sleep(2)
    if not status['tunnel_ready']:
        status['stage'] = 'failed_cloudflared_timeout'
        status['tunnel_log_tail'] = read_tail(TUNNEL_LOG, 16000)
        save_status()
        raise RuntimeError('Cloudflare tunnel did not produce a URL within 180 seconds')

    status['stage'] = 'ready'
    status['message'] = 'Open tunnel_url in your browser. Stop the Kaggle notebook when finished.'
    status['comfy_log_tail'] = read_tail(COMFY_LOG, 8000)
    status['tunnel_log_tail'] = read_tail(TUNNEL_LOG, 8000)
    save_status()
    print('OPEN THIS URL IN YOUR BROWSER:', status['tunnel_url'], flush=True)
    while comfy_process.poll() is None and tunnel_process.poll() is None:
        time.sleep(30)
        status['stage'] = 'ready'
        status['tunnel_log_tail'] = read_tail(TUNNEL_LOG, 4000)
        save_status()
finally:
    stop_process(tunnel_process)
    stop_process(comfy_process)
    comfy_handle.close()
    tunnel_handle.close()
    status['stage'] = 'stopped'
    status['comfy_log_tail'] = read_tail(COMFY_LOG, 12000)
    status['tunnel_log_tail'] = read_tail(TUNNEL_LOG, 12000)
    save_status()
